# 피드백 데이터 층화 분할 (Stratified Split)

고등학교 과학 수업에서 수집된 2,080건의 동료 피드백 텍스트 데이터를 **층화 추출(stratified sampling)** 방식으로 훈련/테스트 세트로 분리합니다.

- **데이터**: 7단계 코딩 체계(0=무관~6=대안 제시)로 분류된 피드백 텍스트
- **분할 비율**: 80:20 (훈련:테스트)
- **재현성**: 3개의 시드(seed) 값(42, 43, 44)으로 각각 분할하여 결과의 안정성 확보
- **층화 추출**: 각 레이블의 비율이 훈련/테스트 세트에서 동일하게 유지됨

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split  # 층화 분할 함수
from pathlib import Path
from datetime import datetime

## 1. 설정

실험의 주요 파라미터를 정의합니다.

- **TRAIN_RATIO**: 훈련 세트 비율 (0.80 = 80%)
- **RANDOM_SEEDS**: 시드(seed)는 난수 생성의 초기값으로, 같은 시드를 사용하면 동일한 결과를 재현할 수 있습니다. 3개의 서로 다른 시드를 사용하여 분할 결과의 안정성을 검증합니다.
- **OUTPUT_DIR**: 분할된 데이터 파일이 저장될 디렉토리

In [ ]:
# 파일 경로 (노트북 기준 상대 경로)
DATA_PATH = Path("../feedback_data.xlsx")

# Split 비율 설정 (train 비율)
# 80:20 -> 0.80, 85:15 -> 0.85
TRAIN_RATIO = 0.80  # 변경 가능: 0.80 또는 0.85

# Random seeds for reproducibility (3개의 seed로 3개의 파일 생성)
# 시드 값이 다르면 데이터가 다르게 섞이므로, 서로 다른 훈련/테스트 세트가 생성됨
RANDOM_SEEDS = [42, 43, 44]

# 출력 디렉토리
OUTPUT_DIR = Path(".")

## 2. 데이터 로드

원본 Excel 파일을 읽어와 데이터의 기본 구조를 확인합니다.
- `id`: 학생ID_검사지유형q질문번호_시퀀스 형태의 고유 식별자
- `feedback_text`: 동료 피드백 텍스트 (한국어)
- `label`: 7단계 코딩 값 (0~6)

## 2. 데이터 로드

In [ ]:
# Label 컬럼 확인 (label 또는 Label)
label_col = 'label' if 'label' in df.columns else 'Label'
print(f"Label 컬럼: {label_col}")
print(f"\nLabel 분포:")
# 각 레이블의 빈도수를 오름차순으로 출력
# 0=무관, 1=확인, 2=동의/칭찬, 3=평가, 4=설명요청, 5=대안제시(근거X), 6=대안제시(근거O)
print(df[label_col].value_counts().sort_index())

## 3. 층화 분할 수행 (Stratified Split)

`sklearn`의 `train_test_split` 함수에 `stratify` 옵션을 사용하여 **층화 추출**을 수행합니다.
- 층화 추출(stratified sampling): 원본 데이터의 레이블 비율을 훈련/테스트 세트에서도 그대로 유지하는 분할 방법
- 각 시드마다 독립적으로 분할을 수행하여 3쌍의 훈련/테스트 파일을 생성합니다.

In [ ]:
# 각 seed별로 Stratified split 수행 및 저장
train_ratio_int = int(TRAIN_RATIO * 100)  # 0.80 → 80 (파일명에 사용)
test_ratio = int((1 - TRAIN_RATIO) * 100)  # 0.20 → 20

print(f"Split 비율: {train_ratio_int}:{test_ratio}")
print(f"Random Seeds: {RANDOM_SEEDS}")
print("=" * 60)

# 결과 저장용
split_results = {}

for seed in RANDOM_SEEDS:
    print(f"\n[Seed {seed}] Stratified Split 수행...")
    
    # Stratified split: stratify 옵션으로 레이블 비율 유지
    train_df, test_df = train_test_split(
        df,
        train_size=TRAIN_RATIO,       # 훈련 세트 비율
        stratify=df[label_col],        # 이 컬럼의 비율을 유지하며 분할
        random_state=seed              # 재현성을 위한 시드 값
    )
    
    # 파일명 생성 (비율과 시드를 포함하여 구분 가능하게)
    train_filename = OUTPUT_DIR / f"train_set_{train_ratio_int}_seed{seed}.xlsx"
    test_filename = OUTPUT_DIR / f"test_set_{train_ratio_int}_seed{seed}.xlsx"
    
    # Excel로 저장
    train_df.to_excel(train_filename, index=False)
    test_df.to_excel(test_filename, index=False)
    
    print(f"  Train: {len(train_df)}개 -> {train_filename.name}")
    print(f"  Test: {len(test_df)}개 -> {test_filename.name}")
    
    # 결과 저장 (이후 검증용)
    split_results[seed] = {
        'train_df': train_df,
        'test_df': test_df,
        'train_file': train_filename,
        'test_file': test_filename
    }

print("\n" + "=" * 60)
print(f"총 {len(RANDOM_SEEDS)}개의 split 완료!")

In [ ]:
# 각 seed별 Label 분포 비교: 원본 vs 훈련 vs 테스트
for seed in RANDOM_SEEDS:
    train_df = split_results[seed]['train_df']
    test_df = split_results[seed]['test_df']
    
    print("=" * 60)
    print(f"[Seed {seed}] Label 분포 비교")
    print("=" * 60)
    
    # 빈도수와 비율(%)을 함께 표시하여 비교
    comparison = pd.DataFrame({
        'Original': df[label_col].value_counts().sort_index(),
        'Train': train_df[label_col].value_counts().sort_index(),
        'Test': test_df[label_col].value_counts().sort_index()
    })
    
    # 각 세트별 비율(%) 계산
    comparison['Original %'] = (comparison['Original'] / len(df) * 100).round(2)
    comparison['Train %'] = (comparison['Train'] / len(train_df) * 100).round(2)
    comparison['Test %'] = (comparison['Test'] / len(test_df) * 100).round(2)
    
    print(comparison)
    print()

## 4. 파일 저장 확인

생성된 파일 목록과 각 파일의 크기를 출력하여 정상 저장 여부를 확인합니다.

In [4]:
# Label 컬럼 확인 (label 또는 Label)
label_col = 'label' if 'label' in df.columns else 'Label'
print(f"Label 컬럼: {label_col}")
print(f"\nLabel 분포:")
print(df[label_col].value_counts().sort_index())

Label 컬럼: label

Label 분포:
label
0     60
1    513
2    285
3    731
4    181
5    277
6     33
Name: count, dtype: int64


## 5. 검증

저장된 파일을 다시 읽어 데이터 무결성을 검증합니다.
- 파일 크기가 원본 분할 결과와 일치하는지 확인
- **훈련/테스트 데이터 누출(data leakage)** 여부 검증: 동일한 ID가 양쪽에 존재하면 안 됨

In [ ]:
# 저장된 파일 검증
print("저장된 파일 검증:")
print("=" * 60)

all_valid = True

for seed in RANDOM_SEEDS:
    train_file = split_results[seed]['train_file']
    test_file = split_results[seed]['test_file']
    
    # 파일을 다시 읽어서 크기 비교
    train_check = pd.read_excel(train_file)
    test_check = pd.read_excel(test_file)
    
    print(f"\n[Seed {seed}]")
    print(f"  Train: {len(train_check)}개 (원본: {len(split_results[seed]['train_df'])}개)")
    print(f"  Test: {len(test_check)}개 (원본: {len(split_results[seed]['test_df'])}개)")
    
    # 중복 확인: 훈련과 테스트에 같은 ID가 존재하면 데이터 누출
    train_ids = set(train_check['id'].tolist()) if 'id' in train_check.columns else set(train_check.index.tolist())
    test_ids = set(test_check['id'].tolist()) if 'id' in test_check.columns else set(test_check.index.tolist())
    overlap = train_ids.intersection(test_ids)  # 교집합 계산
    
    if len(overlap) == 0:
        print(f"  -> Train/Test 중복 없음 (정상)")
    else:
        print(f"  -> 경고: Train/Test 중복 {len(overlap)}개!")
        all_valid = False

print("\n" + "=" * 60)
if all_valid:
    print("모든 파일 검증 완료 (정상)")
else:
    print("일부 파일에 문제가 있습니다.")

**결과 해석:** 모든 시드에서 파일 크기가 정확히 일치하며, 훈련/테스트 간 데이터 중복이 없습니다. 이 분할 데이터는 1차 실험(RQ1: ML, BERT, LLM 비교)에 사용됩니다.

> **참고**: 이 80:20 무작위 분할은 동일 학생의 데이터가 훈련과 테스트에 모두 포함될 수 있습니다. 학생 단위 분리가 필요한 일반화 실험은 `02_structural_split.ipynb`과 `04_student_split.ipynb`에서 수행합니다.

In [6]:
# 각 seed별 Label 분포 비교
for seed in RANDOM_SEEDS:
    train_df = split_results[seed]['train_df']
    test_df = split_results[seed]['test_df']
    
    print("=" * 60)
    print(f"[Seed {seed}] Label 분포 비교")
    print("=" * 60)
    
    comparison = pd.DataFrame({
        'Original': df[label_col].value_counts().sort_index(),
        'Train': train_df[label_col].value_counts().sort_index(),
        'Test': test_df[label_col].value_counts().sort_index()
    })
    
    comparison['Original %'] = (comparison['Original'] / len(df) * 100).round(2)
    comparison['Train %'] = (comparison['Train'] / len(train_df) * 100).round(2)
    comparison['Test %'] = (comparison['Test'] / len(test_df) * 100).round(2)
    
    print(comparison)
    print()

[Seed 42] Label 분포 비교
       Original  Train  Test  Original %  Train %  Test %
label                                                    
0            60     48    12        2.88     2.88    2.88
1           513    410   103       24.66    24.64   24.76
2           285    228    57       13.70    13.70   13.70
3           731    585   146       35.14    35.16   35.10
4           181    145    36        8.70     8.71    8.65
5           277    222    55       13.32    13.34   13.22
6            33     26     7        1.59     1.56    1.68

[Seed 43] Label 분포 비교
       Original  Train  Test  Original %  Train %  Test %
label                                                    
0            60     48    12        2.88     2.88    2.88
1           513    410   103       24.66    24.64   24.76
2           285    228    57       13.70    13.70   13.70
3           731    585   146       35.14    35.16   35.10
4           181    145    36        8.70     8.71    8.65
5           277    222    5

## 4. 파일 저장

In [7]:
# 생성된 파일 목록 출력
print("생성된 파일 목록:")
print("=" * 60)

for seed in RANDOM_SEEDS:
    train_file = split_results[seed]['train_file']
    test_file = split_results[seed]['test_file']
    train_size = len(split_results[seed]['train_df'])
    test_size = len(split_results[seed]['test_df'])
    
    print(f"\n[Seed {seed}]")
    print(f"  - {train_file.name} ({train_size}개)")
    print(f"  - {test_file.name} ({test_size}개)")

생성된 파일 목록:

[Seed 42]
  - train_set_80_seed42.xlsx (1664개)
  - test_set_80_seed42.xlsx (416개)

[Seed 43]
  - train_set_80_seed43.xlsx (1664개)
  - test_set_80_seed43.xlsx (416개)

[Seed 44]
  - train_set_80_seed44.xlsx (1664개)
  - test_set_80_seed44.xlsx (416개)


## 5. 검증

In [8]:
# 저장된 파일 검증
print("저장된 파일 검증:")
print("=" * 60)

all_valid = True

for seed in RANDOM_SEEDS:
    train_file = split_results[seed]['train_file']
    test_file = split_results[seed]['test_file']
    
    train_check = pd.read_excel(train_file)
    test_check = pd.read_excel(test_file)
    
    print(f"\n[Seed {seed}]")
    print(f"  Train: {len(train_check)}개 (원본: {len(split_results[seed]['train_df'])}개)")
    print(f"  Test: {len(test_check)}개 (원본: {len(split_results[seed]['test_df'])}개)")
    
    # 중복 확인
    train_ids = set(train_check['id'].tolist()) if 'id' in train_check.columns else set(train_check.index.tolist())
    test_ids = set(test_check['id'].tolist()) if 'id' in test_check.columns else set(test_check.index.tolist())
    overlap = train_ids.intersection(test_ids)
    
    if len(overlap) == 0:
        print(f"  -> Train/Test 중복 없음 (정상)")
    else:
        print(f"  -> 경고: Train/Test 중복 {len(overlap)}개!")
        all_valid = False

print("\n" + "=" * 60)
if all_valid:
    print("모든 파일 검증 완료 (정상)")
else:
    print("일부 파일에 문제가 있습니다.")

저장된 파일 검증:

[Seed 42]
  Train: 1664개 (원본: 1664개)
  Test: 416개 (원본: 416개)
  -> Train/Test 중복 없음 (정상)

[Seed 43]
  Train: 1664개 (원본: 1664개)
  Test: 416개 (원본: 416개)
  -> Train/Test 중복 없음 (정상)

[Seed 44]
  Train: 1664개 (원본: 1664개)
  Test: 416개 (원본: 416개)
  -> Train/Test 중복 없음 (정상)

모든 파일 검증 완료 (정상)
